# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print summary metadata
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {meta.version}")
print(f"Date published: {meta.datePublished}")
print(f"License: {meta.license}")


## 2. Data Overview
Explore all record sets in the dataset, and review fields and columns using their `@id`s.

In [ ]:
# List all record sets by their `@id` and name
print("Record sets available:")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set['@id']}")
    print(f"    name: {record_set.get('name')}")
    # List fields for each record set
    if 'field' in record_set:
        print(f"    Fields:")
        fields = record_set['field']
        if not isinstance(fields, list):
            fields = [fields]
        for field in fields:
            # field can be a dict or str
            if isinstance(field, dict):
                print(f"      @id: {field['@id']}, name: {field.get('name')}")
            else:
                print(f"      @id: {field}")
    print()

# Save all record set IDs to use later
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

Let's examine the first few records of every record set.

**Note:** Use the `@id` of the record set as the argument to `dataset.records(record_set=...)`.

In [ ]:
for rsid in record_set_ids:
    print(f"=== Sample records from record set: {rsid} ===")
    for idx, record in enumerate(dataset.records(record_set=rsid)):
        if idx >= 3:  # Show up to 3 sample records per set
            break
        print(record)
    print()

## 3. Data Extraction
Let's load records into pandas DataFrames, using each record set's `@id` as key.

In [ ]:
# Extract all record sets into pandas
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    if records:
        dataframes[rsid] = pd.DataFrame(records)
    else:
        dataframes[rsid] = pd.DataFrame()

# Show the columns of each record set
for rsid, df in dataframes.items():
    print(f"Record set {rsid} columns:")
    print(df.columns.tolist())
    print()
# Preview the first rows of the main clinical table if present
first_table_id = None
for rsid, df in dataframes.items():
    if not df.empty:
        first_table_id = rsid
        break
print(f"Displaying first 5 rows of record set {first_table_id}:")
dataframes[first_table_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps:

1. Filter records by a numeric field (e.g., age at first or second diagnosis).
2. Normalize a numeric column.
3. Group by a key attribute (e.g., sex or tumor location).

All field and column names are referenced by their `@id` as per the Croissant dataset schema.

In [ ]:
# Example: identify a numeric field for EDA
df = dataframes[first_table_id]
# Fallback: try guessing a likely numeric column (e.g., 'age_at_diagnosis')
numeric_candidates = [c for c in df.columns if ('age' in c or 'interval' in c or 'year' in c) and pd.api.types.is_numeric_dtype(df[c])]
if not numeric_candidates:
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
numeric_field_id = numeric_candidates[0] if numeric_candidates else df.columns[0]  # pick first as example

print(f"Using numeric field for EDA: {numeric_field_id}")

# Filter records: e.g., age/interval > threshold (use 50 if age field, else 0)
if 'age' in numeric_field_id:
    threshold = 50
else:
    threshold = 0
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce').fillna(0) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric column
col_vals = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
filtered_df[f"{numeric_field_id}_normalized"] = (col_vals - col_vals.mean()) / col_vals.std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a likely categorical attribute, e.g., 'sex' or 'anatomical_location'
cat_candidates = [c for c in df.columns if ('sex' in c or 'location' in c or 'site' in c or 'status' in c or 'group' in c)]
group_field_id = cat_candidates[0] if cat_candidates else df.columns[1]  # pick first reasonable categorical

if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id}, mean of {numeric_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and relationship with group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=15)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot of numeric field by group field (if available)
if group_field_id in df.columns:
    plt.figure(figsize=(7,5))
    sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to explore a clinical tabular FAIR² dataset described by a Croissant schema. 

- We loaded record sets and inspected their fields using their `@id`.
- We extracted clinical tables into pandas, applied numeric filtering, normalization, and simple grouping, referencing all data elements by `@id`.
- Basic visualizations showed data distributions and group comparisons.

This approach ensures reproducible and schema-consistent access to structured data, supporting robust clinical analytics and FAIR data stewardship.